# UAV-VisLoc Benchmark

Runs all feature-matching pipelines on the UAV-VisLoc dataset on Kaggle.

**Code:** cloned from GitHub (`benchmarking` branch) into `/kaggle/working/UAV_Localization`
**Dataset:** Kaggle Dataset mounted read-only at `/kaggle/input/datasets/youssefelsayed30/uav-visloc/`
**Limit:** 100 images per run (change `--limit` in each cell)
**Success threshold:** 25 m GPS error
**Results & visualizations:** saved to `/kaggle/working/results/`

**Run the setup cell first.** After that, each section is independent.

## Setup — clone repo + define shared paths

Run this once at the start of the session.

In [1]:
print('>>> EDIT MARKER v3 — if you see this, edits reached Kaggle <<<')
print('lmaooo')
import os, sys, subprocess, shutil

REPO_URL    = 'https://github.com/YousefAyman005/UAV_Localization.git'
REPO_BRANCH = 'benchmarking'
REPO        = '/kaggle/working/UAV_Localization'
DATA        = '/kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example'
OUT         = '/kaggle/working/results'

env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            f'cmd failed ({r.returncode}): {" ".join(cmd)}\n'
            f'STDOUT: {r.stdout}\nSTDERR: {r.stderr}'
        )
    return r

if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    shutil.rmtree(REPO)

if not os.path.isdir(REPO):
    _run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, REPO])
else:
    _run(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', REPO_BRANCH])
    _run(['git', '-C', REPO, 'checkout', REPO_BRANCH])
    _run(['git', '-C', REPO, 'reset', '--hard', f'origin/{REPO_BRANCH}'])

os.makedirs(OUT, exist_ok=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(f'REPO: {REPO}')
print(f'DATA: {DATA}')
print(f'OUT:  {OUT}')

>>> EDIT MARKER v3 — if you see this, edits reached Kaggle <<<
lmaooo
REPO: /kaggle/working/UAV_Localization
DATA: /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example
OUT:  /kaggle/working/results


---
## Section 1 — Baseline (OpenCV)

Classical feature detectors using OpenCV. No GPU required, no extra installs.

| Method | Detector | Descriptor | Matcher |
|--------|----------|------------|---------|
| SIFT   | DoG keypoints | 128-d float SIFT | FLANN + Lowe ratio 0.75 |
| ORB    | FAST keypoints | 256-bit binary ORB | BFMatcher Hamming |
| BRISK  | AGAST keypoints | 512-bit binary BRISK | BFMatcher Hamming |

### 1a — Baseline SIFT

Scale-Invariant Feature Transform. Most accurate of the three classical methods. Slower than ORB/BRISK.

In [ ]:
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)

import Baseline_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/baseline_sift_results.csv'
pl.VIZ_DIR   = f'{OUT}/baseline_sift_viz'

sys.argv = ['', '--limit', '100', '--method', 'sift', '--dist', '25', '--visualize']
pl.main()

Loading /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example/03/satellite03.tif ... 35092x24308 px
  Method: SIFT | Preprocessing: none | Dist: 25.0m | 100 images



100%|██████████| 100/100 [04:17<00:00,  2.57s/img]


  Results saved to /kaggle/working/results/baseline_sift_results.csv
  Success (≤25.0m):    34/100 (34.0%)
  Homography found:       48/100 (48.0%)
  Incorrect matches:      14/48 (29.2%) — offset > 25.0m
  Offset (successes):     mean 14.9m  median 14.9m  max 23.8m
  Median inliers: 9 | ratio: 0.455


### 1b — Baseline ORB

Oriented FAST and Rotated BRIEF. Very fast binary descriptor. Less accurate than SIFT but runs in real time.

In [ ]:
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)

import Baseline_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/baseline_orb_results.csv'
pl.VIZ_DIR   = f'{OUT}/baseline_orb_viz'

sys.argv = ['', '--limit', '100', '--method', 'orb', '--dist', '25', '--visualize']
pl.main()

### 1c — Baseline BRISK

Binary Robust Invariant Scalable Keypoints. Similar speed to ORB with a larger 512-bit descriptor. Often more robust to scale changes than ORB.

In [ ]:
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)

import Baseline_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/baseline_brisk_results.csv'
pl.VIZ_DIR   = f'{OUT}/baseline_brisk_viz'

sys.argv = ['', '--limit', '100', '--method', 'brisk', '--dist', '25', '--visualize']
pl.main()

---
## Section 2 — LightGlue

Learned keypoint matcher that works with multiple front-end detectors. Uses an attention-based GNN to prune ambiguous matches. GPU strongly recommended.

| Variant | Detector | Descriptor | Notes |
|---------|----------|------------|-------|
| DISK    | DISK (learned) | DISK (256-d) | Best accuracy |
| SIFT    | DoG | SIFT (128-d) | Good balance |
| DeDoDe-B | DeDoDe (learned) | DeDoDe (256-d) | Strongest detector |

### 2a — LightGlue + DISK

DISK detector with LightGlue matcher. Typically the strongest LightGlue variant for outdoor aerial imagery.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/cvg/LightGlue.git'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)

import lightglue_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/lightglue_disk_results.csv'
pl.VIZ_DIR   = f'{OUT}/lightglue_disk_viz'

sys.argv = ['', '--limit', '100', '--method', 'disk',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()


TESTTTTTTTBLAHHBLHA
  Device: cuda
  Loading models (disk) ... Loaded LightGlue model
done
Loading /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example/03/satellite03.tif ... 35092x24308 px
  Method: DISK | CLAHE: False | Conf: 0.0 | RANSAC: 10.0 | MinInl: 6 | Dist: 25.0m | 100 images



  0%|          | 0/100 [00:00<?, ?img/s]


KeyboardInterrupt: 

### 2b — LightGlue + SIFT

Classic SIFT detector fed into the LightGlue matcher. Good fallback when DISK/DeDoDe weights are unavailable or slow to download.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/cvg/LightGlue.git'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)

import lightglue_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/lightglue_sift_results.csv'
pl.VIZ_DIR   = f'{OUT}/lightglue_sift_viz'

sys.argv = ['', '--limit', '100', '--method', 'sift',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

### 2c — LightGlue + DeDoDe-B

DeDoDe detector (trained to detect repeatable keypoints across viewpoints) with LightGlue. Often finds more matches in low-texture regions than SIFT or DISK.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/cvg/LightGlue.git'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)

import lightglue_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/lightglue_dedodeb_results.csv'
pl.VIZ_DIR   = f'{OUT}/lightglue_dedodeb_viz'

sys.argv = ['', '--limit', '100', '--method', 'dedodeb',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

---
## Section 3 — LoFTR

**LoFTR** (Detector-Free Local Feature Matching with Transformers) matches dense pixel pairs directly without detecting keypoints first. Uses a coarse-to-fine Transformer architecture trained on MegaDepth (`outdoor` weights). Strong in low-texture and repetitive regions where keypoint detectors struggle. GPU required for reasonable speed.

### 3a — LoFTR (outdoor)

Pretrained on MegaDepth outdoor scenes. Correct weight set for UAV/satellite aerial matching.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kornia', '-q'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('loftr_pipeline', None)
sys.modules.pop('visloc_utils', None)

import loftr_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/loftr_results.csv'
pl.VIZ_DIR   = f'{OUT}/loftr_viz'

sys.argv = ['', '--limit', '100', '--pretrained', 'outdoor',
            '--conf', '0.0', '--dist', '25', '--visualize']
pl.main()

  Device: cuda
  Loading LoFTR (outdoor) ... Downloading: "http://cmp.felk.cvut.cz/~mishkdmy/models/loftr_outdoor.ckpt" to /root/.cache/torch/hub/checkpoints/loftr_outdoor.ckpt


100%|██████████| 44.2M/44.2M [00:02<00:00, 18.8MB/s]

done
Loading /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example/03/satellite03.tif ... 

35092x24308 px
  Method: LoFTR (outdoor) | Conf: 0.0 | Dist: 25.0m | 100 images



100%|██████████| 100/100 [04:42<00:00,  2.82s/img]


  Results saved to /kaggle/working/results/loftr_results.csv
  Success (≤25.0m):    61/100 (61.0%)
  Homography found:       77/100 (77.0%)
  Incorrect matches:      16/77 (20.8%) — offset > 25.0m
  Offset (successes):     mean 13.5m  median 13.7m  max 24.9m
  Median inliers: 104 | ratio: 0.402


---
## Section 4 — RoMa

**RoMa** (Robust Dense Feature Matching) produces a dense warp field between image pairs using a DINOv2 backbone, then samples correspondence points from it. No keypoint detection step. Typically the most accurate dense matcher for large viewpoint and scale changes. GPU required; slowest of all methods.

Pretrained on MegaDepth (`outdoor`) — correct for aerial/satellite scenes.

### 4a — RoMa (outdoor)

Dense warp-based matching with DINOv2 backbone. Samples 5000 correspondences from the predicted warp field.

In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/Parskatt/RoMa.git@edd1b8b'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('roma_pipeline', None)
sys.modules.pop('visloc_utils', None)

import roma_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/roma_results.csv'
pl.VIZ_DIR   = f'{OUT}/roma_viz'

sys.argv = ['', '--limit', '100', '--pretrained', 'outdoor',
            '--conf', '0.0', '--num-matches', '5000', '--dist', '25', '--visualize']
pl.main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.1 MB/s eta 0:00:00
  Device: cuda
  Loading RoMa (outdoor) ... Downloading: "https://github.com/Parskatt/storage/releases/download/roma/roma_outdoor.pth" to /root/.cache/torch/hub/checkpoints/roma_outdoor.pth


100%|██████████| 425M/425M [00:02<00:00, 165MB/s] 


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:03<00:00, 329MB/s] 
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Using coarse resolution (560, 560), and upsample res (864, 864)
done
Loading /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example/03/satellite03.tif ... 35092x24308 px
  Method: RoMa (outdoor) | CLAHE: False | Conf: 0.0 | NumMatches: 5000 | RANSAC: 5.0px | MinInl: 10 | Dist: 25.0m | 100 images



100%|██████████| 100/100 [15:13<00:00,  9.14s/img]


  Results saved to /kaggle/working/results/roma_results.csv
  Success (≤25.0m):    89/100 (89.0%)
  Homography accepted:    100/100 (100.0%)
  Incorrect matches:      11/100 (11.0%) — offset > 25.0m
  Offset (successes):     mean 13.6m  median 13.8m  max 24.9m
  Median inliers: 4064 | ratio: 0.813


---
## Section 5 — MATCHA

**MATCHA** (CVPR 2025 Highlight) — unified sparse/dense matcher built on Stable Diffusion + DINOv2 features + DISK keypoints. Installs from source; pretrained weights come from a Google Drive link on the MATCHA GitHub (`matcha_pretrained.pth`, ~1.5 GB).

**Setup requirement:** add the weights as a Kaggle Dataset input. Expected path: `/kaggle/input/matcha-weights/matcha_pretrained.pth`. Override `WEIGHTS` below if yours is elsewhere.

### 5a — MATCHA (DISK keypoints, 512×512)

Default config from the MATCHA geometric demo. DISK-style keypoints with the MATCHA fused SD+DINOv2 descriptor.

In [ ]:
import os, sys, subprocess, pathlib

MATCHA_DIR = '/kaggle/working/matcha'
WEIGHTS    = '/kaggle/input/datasets/youssefelsayed30/matcha-weights/matcha_pretrained.pth'

if not pathlib.Path(MATCHA_DIR, '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/nv-dvl/matcha.git', MATCHA_DIR], check=True)

# Patch matcha's dift_sd.py for two Kaggle-specific source-level issues.
# (Diffusers' strict subclass check is handled by a monkey-patch below,
#  not by editing this file - source patching is too brittle w.r.t. reloads.)
dift = pathlib.Path(MATCHA_DIR, 'third_party/dift/dift_sd.py')
txt = dift.read_text()

# 1. stabilityai/stable-diffusion-2-1 is gated; use the community mirror.
if 'stabilityai/stable-diffusion-2-1' in txt:
    txt = txt.replace('stabilityai/stable-diffusion-2-1',
                      'sd2-community/stable-diffusion-2-1')

# 2. xformers isn't installed on Kaggle. The call is a VRAM optimization only.
txt = txt.replace(
    'onestep_pipe.enable_xformers_memory_efficient_attention()',
    '# onestep_pipe.enable_xformers_memory_efficient_attention()  # disabled on Kaggle')

dift.write_text(txt)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/Parskatt/DeDoDe.git'], check=True)

# Force-reinstall a mutually compatible set. Kaggle's pre-baked versions are
# mixed/newer and produce ImportErrors (reset_sessions, is_offline_mode,
# tokenizers version) when a single pinned package is overlaid.
# --force-reinstall rewrites every file; --no-deps stops the resolver from
# re-bumping the others. transformers==4.49.0 requires tokenizers<0.22.
for pkg in ('huggingface-hub==0.30.2',
            'tokenizers==0.21.2',
            'transformers==4.49.0',
            'diffusers==0.33.1'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--force-reinstall', '--no-deps', pkg], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                '-e', MATCHA_DIR], check=True)

# Drop any already-imported copies so the freshly installed files are loaded.
for _mod in list(sys.modules):
    head = _mod.split('.', 1)[0]
    if head in ('huggingface_hub', 'diffusers', 'transformers', 'tokenizers',
                'third_party', 'matcha', 'matcha_pipeline'):
        sys.modules.pop(_mod, None)

if MATCHA_DIR not in sys.path: sys.path.insert(0, MATCHA_DIR)
if REPO       not in sys.path: sys.path.insert(0, REPO)

# Monkey-patch the diffusers pipeline-loader subclass check. In 0.33.1 it
# rejects MyUNet2DConditionModel (defined in third_party.dift.dift_sd) even
# though it correctly subclasses UNet2DConditionModel -> ModelMixin. The
# failing check is advisory: the loader still uses the passed instance either
# way, so downgrading the ValueError to a warning is safe here.
import diffusers.pipelines.pipeline_loading_utils as _pll
if not getattr(_pll.maybe_raise_or_warn, '_kaggle_softened', False):
    _orig_check = _pll.maybe_raise_or_warn
    def _soft_check(*args, **kwargs):
        try:
            return _orig_check(*args, **kwargs)
        except ValueError as e:
            msg = str(e)
            print(f"[kaggle-patch] suppressed diffusers subclass check: "
                  f"{msg[:120]}{'...' if len(msg) > 120 else ''}")
    _soft_check._kaggle_softened = True
    _pll.maybe_raise_or_warn = _soft_check

assert os.path.exists(WEIGHTS), (
    f'MATCHA weights not found at {WEIGHTS}. '
    'Download matcha_pretrained.pth from the Google Drive link on '
    'https://github.com/nv-dvl/matcha and attach it as a Kaggle Dataset.'
)

import matcha_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/matcha_results.csv'
pl.VIZ_DIR   = f'{OUT}/matcha_viz'

sys.argv = ['', '--limit', '2', '--weights', WEIGHTS,
            '--img-size', '512', '--keypoint-method', 'disk',
            '--dist', '25', '--visualize']
pl.main()

Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead
  Device: cuda
  Loading MATCHA (disk, 512x512) ... BaseFeature - config:namespace(topK=4096, upsampling=0, image_size=(512, 512), scale_factor=32, keypoint_method='disk', max_length=None, device='cuda')
Number of parameters corr net: 128


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

namespace(topK=4096, upsampling=0, image_size=(512, 512), scale_factor=32, keypoint_method='disk', max_length=None, device='cuda')
done
Loading /kaggle/input/datasets/youssefelsayed30/uav-visloc/UAV_VisLoc_example/UAV_VisLoc_example/03/satellite03.tif ... 35092x24308 px
  Method: MATCHA (disk) | Size: 512 | CLAHE: False | Conf: 0.0 | RANSAC: 5.0px | MinInl: 10 | Dist: 25.0m | 2 images



  0%|          | 0/2 [00:00<?, ?img/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 315.81 MiB is free. Including non-PyTorch memory, this process has 14.25 GiB memory in use. Of the allocated memory 13.94 GiB is allocated by PyTorch, and 185.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## Section 6 — JamMa

**JamMa** (CVPR 2025) — LoFTR-style semi-dense coarse-to-fine matcher with a ConvNeXt-V2-nano backbone and **Joint-Mamba** blocks in place of LoFTR's self/cross attention. Source-only install from GitHub; weights auto-download from the v0.1 GitHub Release on first run (cached in `~/.cache/torch/hub/checkpoints/`).

**CUDA required** — `mamba-ssm` has no CPU/MPS kernels. Enable a GPU accelerator in the Kaggle runtime before running this cell.


### 6a — JamMa (outdoor)

Default config from the JamMa demo. Coarse stride 8, fine stride 2, sub-pixel fine refinement. `--resize 832` matches `demo/demo.py`; raise it for sharper matches at higher VRAM cost.


In [12]:
import os, sys, subprocess, pathlib

JAMMA_DIR = '/kaggle/working/JamMa'

if not pathlib.Path(JAMMA_DIR, '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/leoluxxx/JamMa.git', JAMMA_DIR], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'einops', 'timm', 'loguru', 'yacs', 'kornia==0.7.0'], check=True)

cuda_env = {**os.environ,
            'CUDA_HOME': '/usr/local/cuda',
            'PATH': f"/usr/local/cuda/bin:{os.environ.get('PATH', '')}"}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'causal-conv1d>=1.4.0'],
               check=True, env=cuda_env)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'mamba-ssm>=2.0.3'],
               check=True, env=cuda_env)

if JAMMA_DIR not in sys.path: sys.path.insert(0, JAMMA_DIR)
if REPO      not in sys.path: sys.path.insert(0, REPO)

sys.modules.pop('jamma_pipeline', None)
sys.modules.pop('visloc_utils',   None)

import jamma_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'
pl.OUT_CSV   = f'{OUT}/jamma_results.csv'
pl.VIZ_DIR   = f'{OUT}/jamma_viz'

sys.argv = ['', '--limit', '100', '--resize', '832', '--conf', '0.2',
            '--dist', '25', '--visualize']
pl.main()


  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d)


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'causal-conv1d>=1.4.0']' returned non-zero exit status 1.

---
## Section 7 — CLIP-variant image retrieval

**Pure image-to-image retrieval** (no homography, no GPS prior). The satellite GeoTIFF is tiled into a gallery; each UAV image is embedded with the same model; top-1 tile center = predicted GPS. Fairest comparison across classic CLIP, GeoCLIP, and SatCLIP, and contrasts "global-embedding retrieval" with the feature-matching pipelines above.

| Model | Backbone | Pretrain domain |
|-------|----------|-----------------|
| CLIP (LAION-2B) | ViT-B/32 | Natural images + captions |
| GeoCLIP | CLIP ViT-L/14 + MLP | Natural images aligned to GPS |
| SatCLIP-ViT16-L40 | ViT-B/16 | Sentinel-2 satellite imagery |

**Defaults:** `--tile-size 1024 --stride 512` (~3100 tiles, ~28 m grid error floor — borderline with `--dist 25`).
**Caching:** embeddings cached per `(model, tile_size, stride)` under `/kaggle/working/results/clip_cache/`.


### 7a — Classic CLIP (OpenAI/LAION)

OpenAI CLIP ViT-B/32 with LAION-2B weights (via `open_clip`). Pretrained on natural images + captions — no domain knowledge of aerial imagery. Baseline for "what does vanilla CLIP retrieve?"


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'open_clip_torch'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('clip_pipeline', None)
sys.modules.pop('visloc_utils',  None)

import clip_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'

sys.argv = ['', '--model', 'clip', '--limit', '100', '--dist', '25',
            '--tile-size', '1024', '--stride', '512',
            '--cache-dir', f'{OUT}/clip_cache',
            '--out-dir', OUT]
pl.main()


### 7b — GeoCLIP

GeoCLIP ImageEncoder (CLIP ViT-L/14 backbone + MLP head, trained to align images with GPS locations). First run downloads ~900 MB of CLIP ViT-L/14 weights from HuggingFace.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'geoclip'], check=True)

if REPO not in sys.path: sys.path.insert(0, REPO)
sys.modules.pop('clip_pipeline', None)
sys.modules.pop('visloc_utils',  None)

import clip_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'

sys.argv = ['', '--model', 'geoclip', '--limit', '100', '--dist', '25',
            '--tile-size', '1024', '--stride', '512',
            '--cache-dir', f'{OUT}/clip_cache',
            '--out-dir', OUT]
pl.main()


### 7c — SatCLIP

Microsoft SatCLIP-ViT16-L40 — CLIP-style model whose image encoder was pretrained on Sentinel-2 satellite imagery. Most likely of the three to match the aerial-domain features of UAV/satellite tiles.

**Setup:** SatCLIP source is cloned from GitHub; weights download from HuggingFace on first run into `/kaggle/working/satclip_weights/`.


In [ ]:
import os, sys, subprocess, pathlib

SATCLIP_DIR  = '/kaggle/working/satclip'
SATCLIP_CKPT = '/kaggle/working/satclip_weights/satclip-vit16-l40.ckpt'

if not pathlib.Path(SATCLIP_DIR, '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/microsoft/satclip.git', SATCLIP_DIR], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pytorch-lightning', 'torchgeo', 'rasterio'], check=True)

os.makedirs(os.path.dirname(SATCLIP_CKPT), exist_ok=True)
if not os.path.isfile(SATCLIP_CKPT):
    from huggingface_hub import hf_hub_download
    fetched = hf_hub_download(repo_id='microsoft/SatCLIP-ViT16-L40',
                              filename='satclip-vit16-l40.ckpt',
                              local_dir=os.path.dirname(SATCLIP_CKPT))
    if fetched != SATCLIP_CKPT:
        import shutil; shutil.copy(fetched, SATCLIP_CKPT)

if SATCLIP_DIR not in sys.path: sys.path.insert(0, SATCLIP_DIR)
if REPO        not in sys.path: sys.path.insert(0, REPO)

sys.modules.pop('clip_pipeline', None)
sys.modules.pop('visloc_utils',  None)

import clip_pipeline as pl

pl.BASE      = f'{DATA}/03'
pl.SAT_TIF   = f'{pl.BASE}/satellite03.tif'
pl.DRONE_DIR = f'{pl.BASE}/drone'
pl.DRONE_CSV = f'{pl.BASE}/03.csv'
pl.SAT_CSV   = f'{DATA}/satellite_ coordinates_range.csv'

sys.argv = ['', '--model', 'satclip', '--limit', '100', '--dist', '25',
            '--tile-size', '1024', '--stride', '512',
            '--satclip-ckpt', SATCLIP_CKPT,
            '--cache-dir', f'{OUT}/clip_cache',
            '--out-dir', OUT]
pl.main()
